Notebook to calculate RFM scores and segments for the clean marketing dataset, used in Tableau dashboard (see README.md for link). 

In [0]:
import pandas as pd
import numpy as np
from pathlib import Path

In [0]:
base_dir = Path.cwd()  
csv_path = base_dir.parent / "data" / "processed" / "marketing_clean.csv"
marketing_clean = pd.read_csv(csv_path)

In [0]:
marketing_clean[['Recency', 'TotalSpend', 'NumPurchases']].describe()

Each row represents one customer, with spend and purchases summed over the last two years.  No further granularity is available for transactions, so I can't set up a custom lookback window (as I normally would in practice).  The maximum `Recency` is 99 days, so the most dormant customer in this dataset last purchased just over 3 months ago.  

In real life, recency increases constantly for every inactive customer and consequently the range of values for recency is typically always increasing, so ideally a recency score should take into account typical time windows between purchases.  

Howver, `Recency` is fairly evenly distributed in this dataset (see `marketing_cleaning.ipynb`), so here I am scoring simply, based on three equal percentiles.  

In [0]:
marketing_rfm = marketing_clean.copy()
marketing_rfm['Frequency'] = marketing_rfm['NumPurchases']
marketing_rfm['Monetary'] = marketing_rfm['TotalSpend']

marketing_rfm['R'] = pd.qcut(marketing_rfm['Recency'], q=3, labels=[3, 2, 1]).astype(int)
marketing_rfm['F'] = pd.qcut(marketing_rfm['Frequency'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
marketing_rfm['M'] = pd.qcut(marketing_rfm['Monetary'], q=5, labels=[1, 2, 3, 4, 5]).astype(int)
marketing_rfm['FM_minus_2'] = marketing_rfm['F'] + marketing_rfm['M'] - 2 # so range of F+M is 0-8, not 2-10, making string/regex easier # factor out later

marketing_rfm['RFM_score'] = marketing_rfm['R'].astype(str) + marketing_rfm['F'].astype(str) + marketing_rfm['M'].astype(str)
marketing_rfm['RFplusM_score'] = marketing_rfm['R'].astype(str) + marketing_rfm['FM_minus_2'].astype(str)  # factor out later


Assign segments

In [0]:
segment_map = {
    r'^3[45][45]$': 'Champions', 
    r'^3[345][45]$': 'High-potential growth',
    r'^3[45]3$': 'High-potential growth',
    r'^3[345]3$': 'Loyal',
    r'^3[12][1-5]$': 'Recents',
    r'^33[12]$': 'Recents',
    r'^1[1-5][45]$': 'Critical retention',
    r'^1[1-5][1-3]$': 'Low-priority churn',
    r'^2[1-5][45]$': 'High value at risk',
    r'^2[1-5][1-3]$': 'Standard base'
}

marketing_rfm['Segment'] = marketing_rfm['RFM_score'].replace(segment_map, regex=True)

marketing_rfm['Segment'] = marketing_rfm['Segment'].fillna('Default')

print(marketing_rfm['Segment'].value_counts())



In [0]:
out_path = base_dir.parent / "data" / "processed" / "marketing_rfm.csv"
marketing_rfm.to_csv(out_path, index=False)